In [1]:
import pandas as pd
import numpy as np 
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
import nltk 
nltk.download("stopwords")

A DataFrame Creation

In [ ]:
import pandas as pd

df1 = pd.read_csv("genuine_accounts.csv/tweets.csv", encoding="ISO-8859-1")
df2 = pd.read_csv("social_spambots_1.csv/tweets.csv", encoding="ISO-8859-1")
df3 = pd.read_csv("social_spambots_2.csv/tweets.csv", encoding="ISO-8859-1")
df4 = pd.read_csv("social_spambots_3.csv/tweets.csv", encoding="ISO-8859-1")
df5 = pd.read_csv("traditional_spambots_1.csv/tweets.csv", encoding="ISO-8859-1")


In [ ]:
df1['label'] = 0 #genuine
df2['label'] = 1 #bots
df3['label'] = 1
df4['label'] = 1
df5['label'] = 1

In [ ]:
df_merged=pd.concat([df1,df2,df3,df4],ignore_index=True)

In [ ]:
df_merged["text"] = df_merged["text"].fillna("")

df_user = df_merged.groupby("user_id")["text"]\
    .apply(lambda x: " ".join(x))\
    .reset_index()

labels = df_merged.groupby("user_id")["label"].first().reset_index()

df = df_user.merge(labels, on="user_id")

print(df.head())

In [ ]:
df.to_csv("NewDataset.csv")

To ReUse the Merged dataset

In [ ]:
df=pd.read_csv("NewDataset.csv")

text cleaning

In [ ]:
stop_words = set(stopwords.words("english"))
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)
df['cleaned_text']=[clean_text(i) for i in df['text']]

In [ ]:
df.head()

In [ ]:
df.to_csv("CleanedDataset.csv")

Train Test Split

In [2]:
df=pd.read_csv("CleanedDataset.csv")

In [3]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['cleaned_text'],df['label'],test_size=0.2,random_state=42)

Word Embedding

In [ ]:
from gensim.models import Word2Vec
sentences=[text.split() for text in X_train]

model=Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1
)

In [ ]:
model.save("word2vec.model")

Tokenize and Padding

In [4]:
from gensim.models import Word2Vec
model=Word2Vec.load("word2vec.model")

In [ ]:
tokenizer = Tokenizer(num_words=4000)

tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad=pad_sequences(X_train_seq,maxlen=50)
X_test_pad=pad_sequences(X_test_seq,maxlen=50)

In [ ]:
import pickle
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [8]:
import pickle
with open("tokenizer.pkl","rb") as f:
    tokenizer=pickle.load(f)

In [9]:
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad=pad_sequences(X_train_seq,maxlen=50)
X_test_pad=pad_sequences(X_test_seq,maxlen=50)

In [10]:
np.save("X_train_pad.npy",X_train_pad)
np.save("X_test_pad.npy",X_test_pad)

Balancing Dataset

In [11]:
from imblearn.combine import SMOTEENN
smote=SMOTEENN(random_state=42)
X_bal,y_bal=smote.fit_resample(X_train_pad,y_train)


In [12]:
np.save("X_train_bal.npy",X_bal)
np.save("y_train_bal.npy",y_bal)

In [13]:
np.save("X_test.npy",X_test_pad)
np.save("y_test.npy",y_test)

Embedding Matrix

In [14]:
import numpy as np

embedding_dim = 100
max_vocab = 5000

word_index = tokenizer.word_index

embedding_matrix = np.zeros((max_vocab, embedding_dim))

for word, i in word_index.items():
    
    if i >= max_vocab:
        continue
        
    if word in model.wv:
        embedding_matrix[i] = model.wv[word]
    else:
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))

In [15]:
np.save("embedding_matrix.npy", embedding_matrix)